# KDE envelope grid resolution: how coarse can we go?

`build_kde_envelopes()` in `Source_Analysis/filter_fields.py` builds a rolling
KDE of $\Delta m = m_{\rm ZTF} - m_{\rm PS1}$ against magnitude, on two grids:

| parameter | production default | points |
|---|---|---|
| `dmag_grid` | `np.linspace(-5, 5, 1000)` | 1000 (0.010 mag spacing) |
| `mag_grid`  | `np.arange(14, 24, 0.25)`  | 40 (0.25 mag spacing) |

This notebook varies **each one independently** and asks what it costs us.

**What "cost" means here.** The KDE curves are never used directly. What the
pipeline consumes is `Envelope_KDE.get_percentile()` at the
$\pm3\sigma$ percentiles (0.135% and 99.865%), which `only_big_dmag()` uses as
the keep/reject boundary. So the figures below compare **the $\pm3\sigma$
boundary in $\Delta m$**, not the density curves, and also report the fraction
of real sources whose keep/reject decision actually flips.

Two things to watch for:

1. Those are deep-tail percentiles, and the CDF is a plain `cumsum` over
   `dmag_grid`. Coarsening that grid coarsens the tail integration, which is
   where the boundary lives.
2. `half_bin_width=0.5` means each magnitude bin is 1.0 mag wide while the grid
   steps 0.25 mag, so adjacent bins already share ~75% of their sources. Much of
   the `mag_grid` work is redundant by construction.


In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy.stats import norm
from KDEpy import TreeKDE, FFTKDE

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../..'))
from long_transients.Source_Analysis.filter_fields import Envelope_KDE

REPO = os.path.abspath('..')
QUAD = os.path.join(REPO, 'Data/catalog_results_test/000791_01_1')
BANDS = ['g', 'r', 'i']

# Production defaults, from build_kde_envelopes()
MAG_RANGE       = (14, 24)
MAG_STEP_PROD   = 0.25
DMAG_RANGE      = (-5.0, 5.0)
N_DMAG_PROD     = 1000
HALF_BIN_WIDTH  = 0.5
MIN_DMAGERR     = 0.01

# What only_big_dmag() actually asks the envelope for
SIGMA_BOUNDARY = 3.0
PCTS = [norm.cdf(-SIGMA_BOUNDARY), norm.cdf(SIGMA_BOUNDARY)]

# Dense magnitude axis on which envelopes are compared
MAG_EVAL = np.arange(14.5, 23.51, 0.05)

N_REPEAT = 3   # build-time repeats; we report the best
print(f'comparing the {PCTS[0]*100:.3f}% / {PCTS[1]*100:.3f}% boundaries')


## 1. Load the envelope's input sample

Exactly the sample `build_kde_envelopes()` sees: `Catalog_Flag == 0` (detected in
both catalogs), finite PSF mags in both, with the same `mag`/`dmag`/`dmagerr`
definitions and the same `min_dmagerr` floor.


In [ ]:
def load_band_points(quad_dir, band):
    tab = Table.read(os.path.join(quad_dir, f'{band}_associated.ecsv'))
    tab = tab[tab['Catalog_Flag'] == 0]

    pm = np.array(tab[f'PSTARR_{band}PSFMag'], float)
    pe = np.array(tab[f'PSTARR_{band}PSFMagErr'], float)
    zm = np.array(tab[f'ZTF_{band}PSFMag'], float)
    ze = np.array(tab[f'ZTF_{band}PSFMagErr'], float)

    keep = np.isfinite(pm) & np.isfinite(zm)
    pm, pe, zm, ze = pm[keep], pe[keep], zm[keep], ze[keep]

    mag     = np.mean([pm, zm], axis=0)
    dmag    = zm - pm
    dmagerr = np.maximum(np.sqrt(pe**2 + ze**2), MIN_DMAGERR)

    ok = np.isfinite(dmag) & np.isfinite(dmagerr)
    return mag[ok], dmag[ok], dmagerr[ok]


POINTS = {b: load_band_points(QUAD, b) for b in BANDS}
for b, (mag, dmag, dmagerr) in POINTS.items():
    print(f'{b}: {len(mag):6d} sources   mag {mag.min():.1f}-{mag.max():.1f}   '
          f'median dmagerr {np.median(dmagerr):.3f}')


## 2. Rebuild the envelope at arbitrary grids

`build_envelope()` mirrors the body of `build_kde_envelopes()` for one band, but
takes pre-extracted arrays so we can re-grid without re-running the pipeline.
The returned object is the **production** `Envelope_KDE`, so `get_percentile()`
is the real code path.


In [ ]:
def build_envelope(mag, dmag, dmagerr, mag_grid, dmag_grid,
                   half_bin_width=HALF_BIN_WIDTH):
    '''Single-band equivalent of build_kde_envelopes().'''
    rows = []
    for mag_center in mag_grid:
        m = np.abs(mag - mag_center) < half_bin_width
        if m.sum() <= 1:
            rows.append(np.full(len(dmag_grid), np.nan))
            continue
        rows.append(TreeKDE(bw=dmagerr[m]).fit(dmag[m]).evaluate(dmag_grid))
    return Envelope_KDE(mag_grid, dmag_grid, np.array(rows))


def timed_build(points, mag_grid, dmag_grid, n_repeat=N_REPEAT):
    '''Build, returning (envelope, best-of-n wall time in seconds).'''
    best = np.inf
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        env = build_envelope(*points, mag_grid=mag_grid, dmag_grid=dmag_grid)
        best = min(best, time.perf_counter() - t0)
    return env, best


def make_mag_grid(step):   return np.arange(MAG_RANGE[0], MAG_RANGE[1], step)
def make_dmag_grid(n):     return np.linspace(DMAG_RANGE[0], DMAG_RANGE[1], n)


### Baseline — the production configuration


In [ ]:
BASE_MAG_GRID  = make_mag_grid(MAG_STEP_PROD)
BASE_DMAG_GRID = make_dmag_grid(N_DMAG_PROD)

baseline, base_time = {}, {}
for b in BANDS:
    baseline[b], base_time[b] = timed_build(POINTS[b], BASE_MAG_GRID, BASE_DMAG_GRID)
    print(f'{b}: {base_time[b]:.2f} s   '
          f'({len(BASE_MAG_GRID)} mag bins x {len(BASE_DMAG_GRID)} dmag points)')
print(f'\ntotal baseline build: {sum(base_time.values()):.2f} s for {len(BANDS)} bands')


## 3. Scoring a configuration

Three numbers per band:

- **boundary shift** — $|\Delta m_{\pm3\sigma}^{\rm test} - \Delta m_{\pm3\sigma}^{\rm base}|$
  in millimag, over `MAG_EVAL`. This is the quantity the filter compares against.
- **decision flips** — fraction of real sources in the band whose keep/reject
  verdict changes. The number that actually matters operationally.
- **coverage** — fraction of `MAG_EVAL` where the test envelope is finite but the
  baseline was too. A coarse `mag_grid` can drop a sparse bin to NaN and blank out
  a whole magnitude range through the interpolation.


In [ ]:
def score(env, base_env, points):
    lo,  hi  = env.get_percentile(MAG_EVAL, PCTS)
    blo, bhi = base_env.get_percentile(MAG_EVAL, PCTS)

    base_ok  = np.isfinite(blo) & np.isfinite(bhi)
    resid    = np.concatenate([(lo - blo)[base_ok], (hi - bhi)[base_ok]]) * 1e3  # mmag
    coverage = np.isfinite(lo[base_ok] + hi[base_ok]).mean()

    mag, dmag, _ = points
    v, bv = env.get_percentile(mag, PCTS), base_env.get_percentile(mag, PCTS)
    both  = np.isfinite(v[0] + v[1] + bv[0] + bv[1])
    out   = (dmag < v[0])  | (dmag > v[1])
    bout  = (dmag < bv[0]) | (dmag > bv[1])
    flip  = (out != bout)[both].mean()

    return dict(
        med_shift = np.nanmedian(np.abs(resid)),
        max_shift = np.nanmax(np.abs(resid)),
        coverage  = coverage,
        flip_frac = flip,
        n_kept    = int(out[both].sum()),
        n_kept_base = int(bout[both].sum()),
        n_lost    = int((bout & ~out)[both].sum()),   # candidate -> rejected
        n_gained  = int((~bout & out)[both].sum()),   # rejected  -> candidate
        lo=lo, hi=hi, blo=blo, bhi=bhi,
    )


## 4. Sweep A — $\Delta m$ grid resolution (`mag_grid` held at 0.25)


In [ ]:
N_DMAG_SWEEP = [1000, 500, 250, 100, 50, 25]

sweepA = {}
for n in N_DMAG_SWEEP:
    dgrid = make_dmag_grid(n)
    per_band, tsum = {}, 0.0
    for b in BANDS:
        env, dt = timed_build(POINTS[b], BASE_MAG_GRID, dgrid)
        per_band[b] = score(env, baseline[b], POINTS[b])
        per_band[b]['env'] = env
        tsum += dt
    sweepA[n] = dict(bands=per_band, time=tsum,
                     spacing=(DMAG_RANGE[1] - DMAG_RANGE[0]) / (n - 1))
    print(f'N_dmag={n:5d}  spacing={sweepA[n]["spacing"]:.4f} mag  '
          f'build={tsum:5.2f}s  '
          f'max shift={max(per_band[b]["max_shift"] for b in BANDS):7.1f} mmag  '
          f'flips={max(per_band[b]["flip_frac"] for b in BANDS)*100:5.2f}%')


## 5. Sweep B — `mag_grid` spacing (`dmag_grid` held at 1000)


In [ ]:
MAG_STEP_SWEEP = [0.25, 0.5, 1.0, 2.0]

sweepB = {}
for step in MAG_STEP_SWEEP:
    mgrid = make_mag_grid(step)
    per_band, tsum = {}, 0.0
    for b in BANDS:
        env, dt = timed_build(POINTS[b], mgrid, BASE_DMAG_GRID)
        per_band[b] = score(env, baseline[b], POINTS[b])
        per_band[b]['env'] = env
        tsum += dt
    sweepB[step] = dict(bands=per_band, time=tsum, n_bins=len(mgrid))
    print(f'mag step={step:.2f}  n_bins={len(mgrid):3d}  build={tsum:5.2f}s  '
          f'max shift={max(per_band[b]["max_shift"] for b in BANDS):7.1f} mmag  '
          f'flips={max(per_band[b]["flip_frac"] for b in BANDS)*100:5.2f}%  '
          f'coverage={min(per_band[b]["coverage"] for b in BANDS)*100:5.1f}%')


## 6. The envelopes themselves

Top row: $\Delta m$ grid sweep. Bottom row: `mag_grid` sweep. Black dashed is the
production baseline; the grey cloud is the source sample.


In [ ]:
def plot_envelopes(sweep, label_fmt, title, ax_row, cmap='viridis'):
    keys = list(sweep.keys())
    colors = plt.cm.get_cmap(cmap)(np.linspace(0, 0.9, len(keys)))
    for j, b in enumerate(BANDS):
        ax = ax_row[j]
        mag, dmag, _ = POINTS[b]
        ax.plot(mag, dmag, '.', ms=1, color='0.85', zorder=0, rasterized=True)
        for k, c in zip(keys, colors):
            s = sweep[k]['bands'][b]
            ax.plot(MAG_EVAL, s['lo'], color=c, lw=1.4, label=label_fmt.format(k))
            ax.plot(MAG_EVAL, s['hi'], color=c, lw=1.4)
        s0 = sweep[keys[0]]['bands'][b]
        ax.plot(MAG_EVAL, s0['blo'], 'k--', lw=1.0, zorder=5)
        ax.plot(MAG_EVAL, s0['bhi'], 'k--', lw=1.0, zorder=5, label='production')
        ax.set_xlim(14.5, 23.5); ax.set_ylim(-3, 3)
        ax.set_xlabel('mean mag'); ax.set_title(f'{b} band — {title}', fontsize=10)
        if j == 0:
            ax.set_ylabel(r'$\Delta m$  (ZTF $-$ PS1)')
            ax.legend(fontsize=7, loc='upper left', ncol=2)


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_envelopes(sweepA, 'N={}',        r'$\Delta m$ grid', axes[0])
plot_envelopes(sweepB, 'step={:.2f}', 'mag grid',        axes[1], cmap='plasma')
fig.suptitle(r'$\pm3\sigma$ envelope vs grid resolution', y=0.98)
fig.tight_layout()
plt.show()


## 7. Boundary shift relative to production

Same curves, differenced against the baseline. Note the $y$ scale — this is the
error budget that matters, in millimag.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)

for row, (sweep, fmt, title, cmap) in enumerate([
        (sweepA, 'N={}',        r'$\Delta m$ grid', 'viridis'),
        (sweepB, 'step={:.2f}', 'mag grid',         'plasma')]):
    keys = list(sweep.keys())
    colors = plt.cm.get_cmap(cmap)(np.linspace(0, 0.9, len(keys)))
    for j, b in enumerate(BANDS):
        ax = axes[row, j]
        for k, c in zip(keys, colors):
            s = sweep[k]['bands'][b]
            ax.plot(MAG_EVAL, (s['lo'] - s['blo']) * 1e3, color=c, lw=1.3,
                    label=fmt.format(k))
            ax.plot(MAG_EVAL, (s['hi'] - s['bhi']) * 1e3, color=c, lw=1.3, ls='--')
        ax.axhline(0, color='k', lw=0.8)
        ax.set_title(f'{b} band — {title}', fontsize=10)
        ax.set_xlabel('mean mag')
        if j == 0:
            ax.set_ylabel('boundary shift (mmag)')
            ax.legend(fontsize=7, ncol=2)

fig.suptitle(r'Shift in the $\pm3\sigma$ boundary  (solid = lower, dashed = upper)',
             y=0.99)
fig.tight_layout()
plt.show()


## 8. Accuracy against cost

Left column: how far the boundary moves. Middle: how many real sources change
verdict. Right: build time and speed-up over production.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for row, (sweep, xlabel, xs, logx) in enumerate([
        (sweepA, r'$\Delta m$ grid points', list(sweepA.keys()), True),
        (sweepB, 'mag grid spacing (mag)',  list(sweepB.keys()), False)]):

    med = [max(sweep[k]['bands'][b]['med_shift'] for b in BANDS) for k in xs]
    mx  = [max(sweep[k]['bands'][b]['max_shift'] for b in BANDS) for k in xs]
    flp = [max(sweep[k]['bands'][b]['flip_frac'] for b in BANDS) * 100 for k in xs]
    tm  = [sweep[k]['time'] for k in xs]
    base_total = sum(base_time.values())

    ax = axes[row, 0]
    ax.plot(xs, mx,  'o-', label='worst over mag & band')
    ax.plot(xs, med, 's-', label='median')
    ax.axhline(10, color='r', ls=':', lw=1, label='10 mmag')
    ax.set_yscale('log'); ax.set_ylabel('boundary shift (mmag)')
    ax.set_xlabel(xlabel); ax.legend(fontsize=8)
    ax.set_title('accuracy', fontsize=10)

    ax = axes[row, 1]
    ax.plot(xs, flp, 'o-', color='C3')
    ax.set_ylabel('sources changing verdict (%)'); ax.set_xlabel(xlabel)
    ax.set_title('downstream impact', fontsize=10)

    ax = axes[row, 2]
    ax.plot(xs, tm, 'o-', color='C2', label='build time (3 bands)')
    ax.set_ylabel('seconds'); ax.set_xlabel(xlabel)
    ax2 = ax.twinx()
    ax2.plot(xs, [base_total / t for t in tm], '^--', color='C4', label='speed-up')
    ax2.set_ylabel(r'speed-up vs production ($\times$)')
    ax.set_title('cost', fontsize=10)
    lines = ax.get_lines() + ax2.get_lines()
    ax.legend(lines, [l.get_label() for l in lines], fontsize=8, loc='best')

    if logx:
        for a in axes[row]:
            a.set_xscale('log')

axes[0, 0].annotate(r'$\Delta m$ grid sweep', xy=(-0.28, 0.5),
                    xycoords='axes fraction', rotation=90, va='center',
                    fontsize=12, weight='bold')
axes[1, 0].annotate('mag grid sweep', xy=(-0.28, 0.5),
                    xycoords='axes fraction', rotation=90, va='center',
                    fontsize=12, weight='bold')
fig.tight_layout()
plt.show()


### Where the worst shift actually lands

A single global maximum is a poor summary for the `mag_grid` sweep: the extremes
sit at the sparse bright and faint ends of the sample, not where the sources are.
Below, per band and per configuration — the worst shift, the magnitude it occurs
at, the worst shift restricted to the well-populated 16-22 mag range, and the
90th percentile over all magnitudes.


In [ ]:
CORE = (MAG_EVAL > 16) & (MAG_EVAL < 22)
rows = []

for tag, sweep, fmt in [('dmag grid', sweepA, 'N={}'),
                        ('mag grid',  sweepB, 'step={}')]:
    for k, d in sweep.items():
        for b in BANDS:
            s = d['bands'][b]
            shift = np.nanmax(np.abs(np.vstack([s['lo'] - s['blo'],
                                                s['hi'] - s['bhi']])), axis=0) * 1e3
            if np.all(np.isnan(shift)):
                continue
            i = np.nanargmax(shift)
            rows.append(dict(sweep=tag, setting=fmt.format(k), band=b,
                             max_mmag=shift[i], at_mag=MAG_EVAL[i],
                             max_core_mmag=np.nanmax(shift[CORE]),
                             p90_mmag=np.nanpercentile(shift, 90)))

detail = pd.DataFrame(rows)
print('max_core_mmag is restricted to 16 < mag < 22.\n')
print(detail.to_string(index=False, float_format=lambda v: f'{v:8.1f}'))


### Which way do the flips go?

The flip *fraction* in section 8 is taken over every source, and that denominator
badly understates the effect: only a few dozen sources in the whole quadrant land
outside the $\pm3\sigma$ envelope. The number that matters is the flip count
against the **candidate list**, and its direction:

- **lost** (candidate $\rightarrow$ rejected) is completeness gone — a real
  transient dropped off the list.
- **gained** (rejected $\rightarrow$ candidate) is a narrower envelope letting
  more sources through — extra follow-up burden, not extra recall.

These are not interchangeable, and the two parameters do not fail the same way.


In [ ]:
rows = []
for tag, sweep, fmt in [('dmag grid', sweepA, 'N={}'),
                        ('mag grid',  sweepB, 'step={}')]:
    for k, d in sweep.items():
        base_n = sum(d['bands'][b]['n_kept_base'] for b in BANDS)
        lost   = sum(d['bands'][b]['n_lost']     for b in BANDS)
        gained = sum(d['bands'][b]['n_gained']   for b in BANDS)
        rows.append(dict(sweep=tag, setting=fmt.format(k),
                         candidates_base=base_n,
                         candidates_new=sum(d['bands'][b]['n_kept'] for b in BANDS),
                         lost=lost, gained=gained, net=gained - lost,
                         lost_pct=100 * lost / max(base_n, 1),
                         speedup=sum(base_time.values()) / d['time']))

flips = pd.DataFrame(rows)
print('Counts are summed over g, r, i.\n')
print(flips.to_string(index=False, float_format=lambda v: f'{v:7.1f}'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (tag, sweep, xlabel, logx) in zip(axes, [
        ('dmag grid', sweepA, r'$\Delta m$ grid points', True),
        ('mag grid',  sweepB, 'mag grid spacing (mag)',  False)]):
    xs = list(sweep.keys())
    lost   = [sum(sweep[k]['bands'][b]['n_lost']   for b in BANDS) for k in xs]
    gained = [sum(sweep[k]['bands'][b]['n_gained'] for b in BANDS) for k in xs]
    base_n = sum(sweep[xs[0]]['bands'][b]['n_kept_base'] for b in BANDS)

    ax.plot(xs, lost,   'o-', color='C3', label='lost (candidate -> rejected)')
    ax.plot(xs, gained, 's-', color='C0', label='gained (rejected -> candidate)')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel(xlabel); ax.set_ylabel('sources')
    ax.set_title(f'{tag}  (baseline candidate list = {base_n})', fontsize=10)
    ax.legend(fontsize=8)
    if logx:
        ax.set_xscale('log')

fig.suptitle('Flip direction against the candidate list', y=1.00)
fig.tight_layout()
plt.show()


## 9. Why the $\Delta m$ grid bites where it does

The boundary sits at the 0.135% / 99.865% points of a CDF formed by `cumsum`
over `dmag_grid`. Below, one representative magnitude bin on a log density axis,
with the $\pm3\sigma$ crossings marked — the boundary lives out on the tail,
where a coarse grid has very few samples to integrate over.


In [ ]:
band, mag_center = 'r', 20.0
mag, dmag, dmagerr = POINTS[band]
m = np.abs(mag - mag_center) < HALF_BIN_WIDTH

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = plt.cm.get_cmap('viridis')(np.linspace(0, 0.9, len(N_DMAG_SWEEP)))

for n, c in zip(N_DMAG_SWEEP, colors):
    g = make_dmag_grid(n)
    y = TreeKDE(bw=dmagerr[m]).fit(dmag[m]).evaluate(g)
    axes[0].plot(g, y, '.-', ms=3, lw=1, color=c, label=f'N={n}')
    cdf = np.cumsum(y) / np.sum(y)
    for p in PCTS:
        axes[1].plot(n, np.interp(p, cdf, g), 'o', color=c)

axes[0].set_yscale('log'); axes[0].set_xlim(-1.5, 1.5)
axes[0].set_ylim(max(1e-6, axes[0].get_ylim()[0]), None)
axes[0].set_xlabel(r'$\Delta m$'); axes[0].set_ylabel('KDE density')
axes[0].set_title(f'{band} band, mag bin centred {mag_center} '
                  f'({m.sum()} sources)', fontsize=10)
axes[0].legend(fontsize=8)

axes[1].set_xscale('log')
axes[1].set_xlabel(r'$\Delta m$ grid points')
axes[1].set_ylabel(r'$\pm3\sigma$ boundary  ($\Delta m$)')
axes[1].set_title('recovered boundary vs grid resolution', fontsize=10)
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()


## 10. The combined setting: `mag` step 0.5 + `N_dmag` 500

Sections 4 and 5 varied one parameter at a time, so their errors are not
guaranteed to compose. This section builds the actual combined configuration and
checks it directly against production.


In [ ]:
COMBO_MAG_GRID  = make_mag_grid(0.5)
COMBO_DMAG_GRID = make_dmag_grid(500)

combo, combo_time = {}, {}
for b in BANDS:
    env, dt = timed_build(POINTS[b], COMBO_MAG_GRID, COMBO_DMAG_GRID)
    combo[b] = score(env, baseline[b], POINTS[b])
    combo[b]['env'] = env
    combo_time[b] = dt

base_total  = sum(base_time.values())
combo_total = sum(combo_time.values())

lost   = sum(combo[b]['n_lost']      for b in BANDS)
gained = sum(combo[b]['n_gained']    for b in BANDS)
base_n = sum(combo[b]['n_kept_base'] for b in BANDS)
new_n  = sum(combo[b]['n_kept']      for b in BANDS)

# What the independent sweeps would have predicted, if errors simply added
pred_lost   = (sum(sweepA[500][ 'bands'][b]['n_lost']   for b in BANDS) +
               sum(sweepB[0.5]['bands'][b]['n_lost']   for b in BANDS))
pred_gained = (sum(sweepA[500][ 'bands'][b]['n_gained'] for b in BANDS) +
               sum(sweepB[0.5]['bands'][b]['n_gained'] for b in BANDS))

print(f'build time   : {combo_total:.2f} s vs {base_total:.2f} s baseline '
      f'({base_total / combo_total:.1f}x faster)')
print(f'candidates   : {base_n} -> {new_n}')
print(f'lost/gained  : {lost} lost, {gained} gained '
      f'({100 * lost / base_n:.1f}% of the candidate list lost)')
print(f'  if the two effects simply added: {pred_lost} lost, {pred_gained} gained')
print(f'coverage     : {min(combo[b]["coverage"] for b in BANDS) * 100:.1f}%')
print()
print(f'{"band":5s} {"med shift":>10s} {"max shift":>10s} {"max 16-22":>10s} '
      f'{"lost":>5s} {"gained":>7s}')
CORE_ = (MAG_EVAL > 16) & (MAG_EVAL < 22)
for b in BANDS:
    c = combo[b]
    shift = np.nanmax(np.abs(np.vstack([c['lo'] - c['blo'],
                                        c['hi'] - c['bhi']])), axis=0) * 1e3
    print(f'{b:5s} {c["med_shift"]:9.1f}m {c["max_shift"]:9.1f}m '
          f'{np.nanmax(shift[CORE_]):9.1f}m {c["n_lost"]:5d} {c["n_gained"]:7d}')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True)

for j, b in enumerate(BANDS):
    c = combo[b]
    mag, dmag, _ = POINTS[b]

    ax = axes[0, j]
    ax.plot(mag, dmag, '.', ms=1, color='0.85', zorder=0, rasterized=True)
    ax.plot(MAG_EVAL, c['blo'], 'k-',  lw=1.6, label='production')
    ax.plot(MAG_EVAL, c['bhi'], 'k-',  lw=1.6)
    ax.plot(MAG_EVAL, c['lo'],  'r--', lw=1.4, label='step=0.5, N=500')
    ax.plot(MAG_EVAL, c['hi'],  'r--', lw=1.4)
    ax.set_ylim(-3, 3); ax.set_title(f'{b} band', fontsize=11)
    if j == 0:
        ax.set_ylabel(r'$\Delta m$  (ZTF $-$ PS1)'); ax.legend(fontsize=8)

    ax = axes[1, j]
    ax.plot(MAG_EVAL, (c['lo'] - c['blo']) * 1e3, 'C0-',  lw=1.3, label='lower bound')
    ax.plot(MAG_EVAL, (c['hi'] - c['bhi']) * 1e3, 'C1--', lw=1.3, label='upper bound')
    ax.axhline(0, color='k', lw=0.8)
    ax.axhspan(-100, 100, color='0.9', zorder=0)
    ax.set_xlabel('mean mag'); ax.set_xlim(14.5, 23.5)
    if j == 0:
        ax.set_ylabel('shift vs production (mmag)'); ax.legend(fontsize=8)

axes[1, 0].annotate('grey band = $\pm$100 mmag', xy=(0.03, 0.06),
                    xycoords='axes fraction', fontsize=8, color='0.4')
fig.suptitle('Combined setting vs production', y=0.98)
fig.tight_layout()
plt.show()


### 1-D KDEs at fixed magnitudes

The envelope is a percentile of these curves, so this is the thing to eyeball.
Each column is one magnitude bin; top row linear, bottom row log (where the
$\pm3\sigma$ boundary actually lives). Vertical lines mark the recovered
boundaries.

Note what is and is not being tested at each centre. `half_bin_width = 0.5`, so
at 16/18/20/22 both grids select the **same sources** and the only difference is
the evaluation grid. The odd centres in the second figure fall *between*
`step=0.5` grid points, so there the envelope is interpolated — that is where the
coarser `mag_grid` is actually doing work.


In [ ]:
def kde_panel(ax_lin, ax_log, band, mag_center):
    mag, dmag, dmagerr = POINTS[band]
    m = np.abs(mag - mag_center) < HALF_BIN_WIDTH
    if m.sum() <= 1:
        return
    out = {}
    for n, color, ls, lbl in [(N_DMAG_PROD, 'k', '-', 'N=1000'),
                              (500, 'C3', '--', 'N=500')]:
        g = make_dmag_grid(n)
        y = TreeKDE(bw=dmagerr[m]).fit(dmag[m]).evaluate(g)
        cdf = np.cumsum(y) / np.sum(y)
        bounds = [np.interp(p, cdf, g) for p in PCTS]
        out[n] = bounds
        for ax in (ax_lin, ax_log):
            ax.plot(g, y, ls, color=color, lw=1.4, label=lbl)
            for bd in bounds:
                ax.axvline(bd, color=color, ls=ls, lw=1.0, alpha=0.8)
    ax_log.set_yscale('log')
    ax_log.set_ylim(1e-5, None)
    for ax in (ax_lin, ax_log):
        ax.set_xlim(-2.5, 2.5)
    d = [(out[500][i] - out[N_DMAG_PROD][i]) * 1e3 for i in range(2)]
    ax_lin.set_title(f'{band}, mag {mag_center}  ({m.sum()} src)\n'
                     f'boundary shift {d[0]:+.0f} / {d[1]:+.0f} mmag', fontsize=9)


for band in BANDS:
    for centers, tag in [([16.0, 18.0, 20.0, 22.0], 'on both mag grids'),
                         ([16.25, 18.25, 20.25, 22.25], 'between step=0.5 grid points')]:
        fig, axes = plt.subplots(2, len(centers), figsize=(16, 6), sharex=True)
        for j, mc in enumerate(centers):
            kde_panel(axes[0, j], axes[1, j], band, mc)
        axes[0, 0].set_ylabel('density (linear)')
        axes[1, 0].set_ylabel('density (log)')
        axes[0, 0].legend(fontsize=8)
        for ax in axes[1]:
            ax.set_xlabel(r'$\Delta m$')
        fig.suptitle(f'{band} band — 1-D KDE per magnitude bin ({tag})', y=0.99)
        fig.tight_layout()
        plt.show()


In [ ]:
# Boundary comparison at the interpolated magnitudes: does step=0.5 reproduce
# what a dedicated bin at that magnitude would have given?
print(f'{"band":5s} {"mag":>6s} {"production":>22s} {"combined":>22s} {"shift (mmag)":>20s}')
for b in BANDS:
    for mc in [16.25, 18.25, 20.25, 22.25]:
        blo, bhi = baseline[b].get_percentile(np.array([mc]), PCTS)
        clo, chi = combo[b]['env'].get_percentile(np.array([mc]), PCTS)
        print(f'{b:5s} {mc:6.2f} {blo[0]:10.3f} {bhi[0]:10.3f} '
              f'{clo[0]:10.3f} {chi[0]:10.3f} '
              f'{(clo[0]-blo[0])*1e3:9.0f} {(chi[0]-bhi[0])*1e3:9.0f}')


## 11. TreeKDE vs FFTKDE

`FFTKDE` is far faster than `TreeKDE`, but it **cannot be swapped in directly**:

```
FFTKDE(bw=array).fit(x).evaluate(g)
  -> ValueError: The bw must be a number.
```

Production passes `bw=dmagerr[mag_mask]` — a *per-source* bandwidth equal to that
source's combined photometric error. That variable bandwidth is the substance of
the method, not an incidental choice, and `FFTKDE` only accepts a scalar.

There is a workaround that keeps most of it. Group the sources in a magnitude bin
by their `dmagerr`, run one `FFTKDE` per group at that group's bandwidth, and sum
the results weighted by group population. With $k$ groups this costs $k$ FFT
passes and converges on the variable-bandwidth estimate as $k$ grows. $k=1$ is the
naive scalar-bandwidth swap, included as the floor.

One other `FFTKDE` constraint: every data point must lie inside the evaluation
grid, which `[-5, 5]` does not guarantee. The helper below evaluates on a padded
grid at the *same spacing* and interpolates back, so the round trip is lossless
on the target grid.


In [ ]:
def _padded_grid(x, bw_max, target_grid):
    # Equidistant grid at target spacing, wide enough for FFTKDE to accept x.
    step = target_grid[1] - target_grid[0]
    lo = min(target_grid[0],  x.min() - 6 * bw_max)
    hi = max(target_grid[-1], x.max() + 6 * bw_max)
    n  = int(np.ceil((hi - lo) / step)) + 1
    return lo + np.arange(n) * step


def fft_binned_kde(x, bw, grid, n_bw_bins):
    # Variable-bandwidth KDE approximated by n_bw_bins fixed-bandwidth FFTKDEs.
    pg = _padded_grid(x, bw.max(), grid)
    out = np.zeros(len(grid))

    if n_bw_bins == 1:
        edges = np.array([bw.min(), bw.max() + 1e-9])
    else:
        edges = np.quantile(bw, np.linspace(0, 1, n_bw_bins + 1))
        edges[-1] += 1e-9

    for i in range(len(edges) - 1):
        m = (bw >= edges[i]) & (bw < edges[i + 1])
        if m.sum() == 0:
            continue
        y = FFTKDE(bw=float(np.mean(bw[m]))).fit(x[m]).evaluate(pg)
        out += np.interp(grid, pg, y) * m.sum()

    return out / len(x)


def build_envelope_fft(mag, dmag, dmagerr, mag_grid, dmag_grid, n_bw_bins,
                       half_bin_width=HALF_BIN_WIDTH):
    rows = []
    for mag_center in mag_grid:
        m = np.abs(mag - mag_center) < half_bin_width
        if m.sum() <= 1:
            rows.append(np.full(len(dmag_grid), np.nan))
            continue
        rows.append(fft_binned_kde(dmag[m], dmagerr[m], dmag_grid, n_bw_bins))
    return Envelope_KDE(mag_grid, dmag_grid, np.array(rows))


def timed_build_fft(points, mag_grid, dmag_grid, n_bw_bins, n_repeat=N_REPEAT):
    best = np.inf
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        env = build_envelope_fft(*points, mag_grid=mag_grid, dmag_grid=dmag_grid,
                                 n_bw_bins=n_bw_bins)
        best = min(best, time.perf_counter() - t0)
    return env, best


### Benchmark at the production grids

Both estimators on `mag_grid` = 0.25 and `dmag_grid` = 1000, so the only variable
is the KDE backend.


In [ ]:
BW_BINS = [1, 3, 5, 10, 20, 50]

fftsweep = {}
for k in BW_BINS:
    per_band, tsum = {}, 0.0
    for b in BANDS:
        env, dt = timed_build_fft(POINTS[b], BASE_MAG_GRID, BASE_DMAG_GRID, k)
        per_band[b] = score(env, baseline[b], POINTS[b])
        per_band[b]['env'] = env
        tsum += dt
    fftsweep[k] = dict(bands=per_band, time=tsum)

CORE = (MAG_EVAL > 16) & (MAG_EVAL < 22)
base_total = sum(base_time.values())

rows = []
for k, d in fftsweep.items():
    core_max = 0.0
    for b in BANDS:
        s = d['bands'][b]
        sh = np.nanmax(np.abs(np.vstack([s['lo'] - s['blo'],
                                         s['hi'] - s['bhi']])), axis=0) * 1e3
        core_max = max(core_max, np.nanmax(sh[CORE]))
    rows.append(dict(
        backend=('FFTKDE scalar bw' if k == 1 else f'FFTKDE {k} bw bins'),
        med_mmag=max(d['bands'][b]['med_shift'] for b in BANDS),
        max_mmag=max(d['bands'][b]['max_shift'] for b in BANDS),
        core_mmag=core_max,
        lost=sum(d['bands'][b]['n_lost'] for b in BANDS),
        gained=sum(d['bands'][b]['n_gained'] for b in BANDS),
        seconds=d['time'], speedup=base_total / d['time']))

fftdf = pd.DataFrame(rows)
print(f'TreeKDE (production): {base_total:.2f} s for 3 bands\n')
print('core_mmag restricted to 16 < mag < 22; lost/gained are candidate counts.\n')
print(fftdf.to_string(index=False, float_format=lambda v: f'{v:8.2f}'))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

ks  = list(fftsweep.keys())
tms = [fftsweep[k]['time'] for k in ks]

ax = axes[0]
ax.plot(ks, [max(fftsweep[k]['bands'][b]['max_shift'] for b in BANDS) for k in ks],
        'o-', label='worst, all mags')
ax.plot(ks, list(fftdf['core_mmag']), 's-', label='worst, 16-22 mag')
ax.plot(ks, [max(fftsweep[k]['bands'][b]['med_shift'] for b in BANDS) for k in ks],
        '^-', label='median')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('bandwidth bins'); ax.set_ylabel('boundary shift vs TreeKDE (mmag)')
ax.set_title('accuracy', fontsize=10); ax.legend(fontsize=8)

ax = axes[1]
ax.plot(ks, [sum(fftsweep[k]['bands'][b]['n_lost']   for b in BANDS) for k in ks],
        'o-', color='C3', label='lost')
ax.plot(ks, [sum(fftsweep[k]['bands'][b]['n_gained'] for b in BANDS) for k in ks],
        's-', color='C0', label='gained')
ax.set_xscale('log'); ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('bandwidth bins'); ax.set_ylabel('sources')
ax.set_title('candidate list (baseline = '
             f'{sum(fftsweep[ks[0]]["bands"][b]["n_kept_base"] for b in BANDS)})',
             fontsize=10)
ax.legend(fontsize=8)

ax = axes[2]
ax.plot(ks, tms, 'o-', color='C2')
ax.axhline(base_total, color='k', ls='--', lw=1.2, label='TreeKDE')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('bandwidth bins'); ax.set_ylabel('build time, 3 bands (s)')
ax.set_title('cost', fontsize=10); ax.legend(fontsize=8)
for k, t in zip(ks, tms):
    ax.annotate(f'{base_total / t:.0f}x', (k, t), textcoords='offset points',
                xytext=(0, -14), fontsize=8, ha='center')

fig.suptitle('FFTKDE (binned bandwidth) against TreeKDE', y=1.00)
fig.tight_layout()
plt.show()


**On picking $k$.** The median shift falls monotonically with more bandwidth bins,
as it should — the approximation converges on the variable-bandwidth estimate. The
*worst-case* shift does not behave as cleanly across bands: it improves in $r$ but
the worst value inside 16-22 mag drifts up with $k$ once the other bands are
included.

That wobble is not grid discretization. Roughly 7% of sources sit at the
`min_dmagerr = 0.01` floor, which is one `dmag_grid` cell, so FFT binning was the
obvious suspect — but rebuilding at 2000 and 4000 $\Delta m$ points reproduces the
max shift to the millimag at every $k$, so the evaluation grid is not involved.
The residual is left unexplained rather than guessed at.

It also does not matter for the decision. Every $k \ge 3$ gives **1 candidate lost,
0 gained** — identical outcomes across the whole range — and all of the mmag-level
movement sits far below the ~0.11 mag median photometric error. $k = 3$ is
therefore the sensible pick: it is the fastest of the converged options at ~110x,
and nothing above it buys a different answer.


### Envelopes and 1-D curves

`K_SHOW` bandwidth bins against production — first as envelopes and residuals,
then as the underlying KDE curves at fixed magnitudes, with the naive scalar-bw
swap included so the difference between the two FFT variants is visible.


In [ ]:
K_SHOW = 3
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True)

for j, b in enumerate(BANDS):
    s = fftsweep[K_SHOW]['bands'][b]
    mag, dmag, _ = POINTS[b]

    ax = axes[0, j]
    ax.plot(mag, dmag, '.', ms=1, color='0.85', zorder=0, rasterized=True)
    ax.plot(MAG_EVAL, s['blo'], 'k-',  lw=1.6, label='TreeKDE (production)')
    ax.plot(MAG_EVAL, s['bhi'], 'k-',  lw=1.6)
    ax.plot(MAG_EVAL, s['lo'],  'r--', lw=1.4, label=f'FFTKDE, {K_SHOW} bw bins')
    ax.plot(MAG_EVAL, s['hi'],  'r--', lw=1.4)
    ax.set_ylim(-3, 3); ax.set_title(f'{b} band', fontsize=11)
    if j == 0:
        ax.set_ylabel(r'$\Delta m$  (ZTF $-$ PS1)'); ax.legend(fontsize=8)

    ax = axes[1, j]
    ax.plot(MAG_EVAL, (s['lo'] - s['blo']) * 1e3, 'C0-',  lw=1.3, label='lower')
    ax.plot(MAG_EVAL, (s['hi'] - s['bhi']) * 1e3, 'C1--', lw=1.3, label='upper')
    ax.axhline(0, color='k', lw=0.8)
    ax.axhspan(-100, 100, color='0.9', zorder=0)
    ax.set_xlabel('mean mag'); ax.set_xlim(14.5, 23.5)
    if j == 0:
        ax.set_ylabel('shift vs TreeKDE (mmag)'); ax.legend(fontsize=8)

fig.suptitle(f'FFTKDE ({K_SHOW} bandwidth bins) vs TreeKDE', y=0.98)
fig.tight_layout()
plt.show()


In [ ]:
CENTERS = [16.0, 18.0, 20.0, 22.0]

def _bounds(y, grid):
    cdf = np.cumsum(y) / np.sum(y)
    return np.array([np.interp(p, cdf, grid) for p in PCTS])

for band in BANDS:
    mag, dmag, dmagerr = POINTS[band]
    fig, axes = plt.subplots(2, len(CENTERS), figsize=(16, 6), sharex=True)
    for j, mc in enumerate(CENTERS):
        m = np.abs(mag - mc) < HALF_BIN_WIDTH
        if m.sum() <= 1:
            continue
        curves = [
            ('TreeKDE', TreeKDE(bw=dmagerr[m]).fit(dmag[m]).evaluate(BASE_DMAG_GRID),
             'k', '-'),
            (f'FFT {K_SHOW} bins',
             fft_binned_kde(dmag[m], dmagerr[m], BASE_DMAG_GRID, K_SHOW), 'C3', '--'),
            ('FFT scalar bw',
             fft_binned_kde(dmag[m], dmagerr[m], BASE_DMAG_GRID, 1), 'C0', ':'),
        ]
        ref = _bounds(curves[0][1], BASE_DMAG_GRID)
        for lbl, y, color, ls in curves:
            bd = _bounds(y, BASE_DMAG_GRID)
            for ax in axes[:, j]:
                ax.plot(BASE_DMAG_GRID, y, ls, color=color, lw=1.4, label=lbl)
                for v in bd:
                    ax.axvline(v, color=color, ls=ls, lw=1.0, alpha=0.8)
        bd = _bounds(curves[1][1], BASE_DMAG_GRID)
        axes[0, j].set_title(f'{band}, mag {mc}  ({m.sum()} src)\n'
                             f'FFT{K_SHOW} shift {(bd[0]-ref[0])*1e3:+.0f} / '
                             f'{(bd[1]-ref[1])*1e3:+.0f} mmag', fontsize=9)
        axes[1, j].set_yscale('log'); axes[1, j].set_ylim(1e-5, None)
        for ax in axes[:, j]:
            ax.set_xlim(-2.5, 2.5)
        axes[1, j].set_xlabel(r'$\Delta m$')
    axes[0, 0].set_ylabel('density (linear)')
    axes[1, 0].set_ylabel('density (log)')
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f'{band} band — KDE backend comparison per magnitude bin', y=0.99)
    fig.tight_layout()
    plt.show()


## 12. Alternatives that keep per-source bandwidths

Binning bandwidths was forced on us by `FFTKDE`'s scalar-`bw` limitation. Three
alternatives avoid it entirely, all using every source's own `dmagerr`:

1. **`NaiveKDE`** — KDEpy's brute-force evaluator. Accepts an array `bw`, so it is
   a literal drop-in for `TreeKDE`.
2. **Direct numpy** — the estimator written out as one broadcast sum of Gaussians,
   $\hat f(g) = \frac{1}{n}\sum_i \phi\!\left(\frac{g - x_i}{b_i}\right)/b_i$.
   No tree, no library; exact for the given grid.
3. **Mixture CDF + root-find** — the one that skips the grid. The estimator *is* a
   Gaussian mixture, so its CDF is available in closed form,
   $F(t) = \frac{1}{n}\sum_i \Phi\!\left(\frac{t - x_i}{b_i}\right)$, and the
   $\pm3\sigma$ boundary is just $F^{-1}$ at the two percentiles — a 1-D root-find.

Point 3 is worth dwelling on. `only_big_dmag()` never wants a density curve; it
wants **two numbers per magnitude bin**. Everything the grid does — evaluate 1000
points, `cumsum` them, interpolate the CDF — is machinery for extracting those two
numbers approximately. Solving for them directly is both cheaper and exact, and it
makes `dmag_grid` disappear as a parameter.

Accuracy below is measured against that exact result, which also prices the
discretization error the production path is already carrying.


In [ ]:
from scipy.special import ndtr
from scipy.optimize import brentq
from KDEpy import NaiveKDE


def direct_grid_kde(x, bw, grid):
    z = (grid[None, :] - x[:, None]) / bw[:, None]
    return (np.exp(-0.5 * z * z) / (bw[:, None] * np.sqrt(2 * np.pi))).sum(0) / len(x)


def grid_bounds(y, grid):
    cdf = np.cumsum(y) / np.sum(y)
    return np.array([np.interp(p, cdf, grid) for p in PCTS])


def exact_bounds(x, bw):
    # Gaussian-mixture CDF inverted directly - no grid involved.
    lo, hi = x.min() - 12 * bw.max(), x.max() + 12 * bw.max()
    return np.array([brentq(lambda t: ndtr((t - x) / bw).mean() - p, lo, hi,
                            xtol=1e-6) for p in PCTS])


BACKENDS = {
    'TreeKDE + grid (production)':
        lambda x, b: grid_bounds(TreeKDE(bw=b).fit(x).evaluate(BASE_DMAG_GRID),
                                 BASE_DMAG_GRID),
    'NaiveKDE + grid':
        lambda x, b: grid_bounds(NaiveKDE(bw=b).fit(x).evaluate(BASE_DMAG_GRID),
                                 BASE_DMAG_GRID),
    'numpy direct + grid':
        lambda x, b: grid_bounds(direct_grid_kde(x, b, BASE_DMAG_GRID),
                                 BASE_DMAG_GRID),
    'FFTKDE 3 bw bins (binned)':
        lambda x, b: grid_bounds(fft_binned_kde(x, b, BASE_DMAG_GRID, 3),
                                 BASE_DMAG_GRID),
    'mixture CDF + brentq':
        exact_bounds,
}


def run_backend(fn, points, mag_grid, n_repeat=N_REPEAT):
    '''Return (bounds per mag bin, best-of-n seconds).'''
    mag, dmag, dmagerr = points
    best, bounds = np.inf, None
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        out = []
        for mc in mag_grid:
            m = np.abs(mag - mc) < HALF_BIN_WIDTH
            out.append(None if m.sum() <= 1 else fn(dmag[m], dmagerr[m]))
        dt = time.perf_counter() - t0
        if dt < best:
            best, bounds = dt, out
    return bounds, best


In [ ]:
backend_res = {name: {'time': 0.0, 'bounds': {}} for name in BACKENDS}

for name, fn in BACKENDS.items():
    for b in BANDS:
        bounds, dt = run_backend(fn, POINTS[b], BASE_MAG_GRID)
        backend_res[name]['bounds'][b] = bounds
        backend_res[name]['time'] += dt

EXACT = backend_res['mixture CDF + brentq']['bounds']
prod_time = backend_res['TreeKDE + grid (production)']['time']

rows = []
for name, d in backend_res.items():
    worst = 0.0
    for b in BANDS:
        for got, ref in zip(d['bounds'][b], EXACT[b]):
            if got is None or ref is None:
                continue
            worst = max(worst, np.abs(got - ref).max())
    rows.append(dict(backend=name, seconds=d['time'],
                     speedup=prod_time / d['time'],
                     max_mmag_vs_exact=worst * 1e3,
                     per_source_bw=('binned' if 'FFT' in name else 'exact')))

backdf = pd.DataFrame(rows)
print('Accuracy is measured against the exact mixture-CDF boundary.\n')
print(backdf.to_string(index=False, float_format=lambda v: f'{v:9.3f}'))


In [ ]:
# What does the exact boundary change about the candidate list?
print(f'{"band":5s} {"prod candidates":>16s} {"exact candidates":>17s} '
      f'{"lost":>5s} {"gained":>7s}')
tot = [0, 0, 0, 0]
for b in BANDS:
    mag, dmag, _ = POINTS[b]
    ex = EXACT[b]
    ok_bins = np.array([e is not None for e in ex])
    lo_c = np.array([e[0] if e is not None else np.nan for e in ex])
    hi_c = np.array([e[1] if e is not None else np.nan for e in ex])

    lo_s = np.interp(mag, BASE_MAG_GRID[ok_bins], lo_c[ok_bins])
    hi_s = np.interp(mag, BASE_MAG_GRID[ok_bins], hi_c[ok_bins])
    bv = baseline[b].get_percentile(mag, PCTS)

    good = np.isfinite(bv[0] + bv[1])
    out  = ((dmag < lo_s)  | (dmag > hi_s)) & good
    bout = ((dmag < bv[0]) | (dmag > bv[1])) & good
    lost, gained = int((bout & ~out).sum()), int((~bout & out).sum())
    print(f'{b:5s} {int(bout.sum()):16d} {int(out.sum()):17d} {lost:5d} {gained:7d}')
    tot = [tot[0] + int(bout.sum()), tot[1] + int(out.sum()),
           tot[2] + lost, tot[3] + gained]
print(f'{"ALL":5s} {tot[0]:16d} {tot[1]:17d} {tot[2]:5d} {tot[3]:7d}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

names = list(backend_res.keys())
times = [backend_res[n]['time'] for n in names]
errs  = [max(r['max_mmag_vs_exact'], 1e-3) for _, r in backdf.iterrows()]
cols  = ['0.3' if 'production' in n else ('C1' if 'FFT' in n else 'C0') for n in names]

ax = axes[0]
ax.barh(range(len(names)), [prod_time / t for t in times], color=cols)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.set_xscale('log'); ax.set_xlabel('speed-up vs production')
ax.axvline(1, color='k', lw=0.8)
for i, t in enumerate(times):
    ax.annotate(f'{prod_time / t:.0f}x  ({t:.2f}s)', (prod_time / t, i),
                textcoords='offset points', xytext=(4, 0), va='center', fontsize=8)
ax.set_title('cost', fontsize=10); ax.set_xlim(right=ax.get_xlim()[1] * 6)

ax = axes[1]
ax.barh(range(len(names)), errs, color=cols)
ax.set_yticks(range(len(names))); ax.set_yticklabels([])
ax.set_xscale('log'); ax.set_xlabel('max boundary error vs exact (mmag)')
for i, e in enumerate(errs):
    ax.annotate('exact' if e <= 1e-3 else f'{e:.1f}', (e, i),
                textcoords='offset points', xytext=(4, 0), va='center', fontsize=8)
ax.set_title('accuracy', fontsize=10); ax.set_xlim(right=ax.get_xlim()[1] * 8)

fig.suptitle('Backends that keep per-source bandwidths (blue) vs binned (orange)',
             y=1.00)
fig.tight_layout()
plt.show()


### The catch

The mixture-CDF route returns **boundaries, not a density**, so it does not fit
behind `Envelope_KDE` as written — that class caches `kde_ys` on a grid and
`get_percentile()` interpolates a CDF out of it at query time. Going gridless
means caching the two boundary curves instead, which fixes `sigma_boundary` at
build time rather than query time.

That is currently free: `sigma_boundary=3.0` at every call site in
`filter_fields.py` and `Debugging/`, never varied. But it is a real reduction in
flexibility, and the cached `.npz` layout changes, so existing
`Data/kde_envelopes/` caches would need rebuilding.

If that tradeoff is unwanted, **`numpy direct + grid`** keeps the current
interface and cache format exactly, keeps per-source bandwidths, and is
~10x faster than `TreeKDE` — no approximation beyond the grid discretization the
production path already has.


## 13. Summary


In [ ]:
rows = []
base_total = sum(base_time.values())

for n, d in sweepA.items():
    rows.append(dict(sweep='dmag grid', setting=f'N={n}',
                     spacing=f"{d['spacing']:.3f} mag",
                     med_mmag=max(d['bands'][b]['med_shift'] for b in BANDS),
                     max_mmag=max(d['bands'][b]['max_shift'] for b in BANDS),
                     lost=sum(d['bands'][b]['n_lost'] for b in BANDS),
                     gained=sum(d['bands'][b]['n_gained'] for b in BANDS),
                     coverage_pct=min(d['bands'][b]['coverage'] for b in BANDS) * 100,
                     seconds=d['time'], speedup=base_total / d['time']))

for step, d in sweepB.items():
    rows.append(dict(sweep='mag grid', setting=f'step={step}',
                     spacing=f"{d['n_bins']} bins",
                     med_mmag=max(d['bands'][b]['med_shift'] for b in BANDS),
                     max_mmag=max(d['bands'][b]['max_shift'] for b in BANDS),
                     lost=sum(d['bands'][b]['n_lost'] for b in BANDS),
                     gained=sum(d['bands'][b]['n_gained'] for b in BANDS),
                     coverage_pct=min(d['bands'][b]['coverage'] for b in BANDS) * 100,
                     seconds=d['time'], speedup=base_total / d['time']))

summary = pd.DataFrame(rows)
print('All error columns are the WORST case over the three bands.\n')
print(summary.to_string(index=False, float_format=lambda v: f'{v:8.2f}'))


### Reading the table

- **`max_mmag`** is the largest $\pm3\sigma$ boundary displacement anywhere in
  14.5–23.5 mag, in any band. Compare it against the median `dmagerr` printed in
  section 1 — a shift well below the per-source photometric error is not
  meaningfully changing the cut.
- **`lost`** / **`gained`** are counts against the candidate list, summed over
  bands. `lost` is completeness given up; `gained` is extra contamination. Weigh
  these against the baseline candidate count, not against the full source count —
  the candidate list is small, so a flip fraction that looks negligible per-source
  can be a sizeable slice of it.
- **`coverage_pct`** below 100 means a coarse `mag_grid` produced a NaN bin that
  blanked part of the magnitude range — a hard failure, not a small error.
- **`speedup`** is against the full three-band production build.

Envelopes are cached per field under `Data/kde_envelopes/{field}/`, so this cost
is paid once per field, not per filter run. Worth weighing before trading
accuracy for it.
